<a href="https://colab.research.google.com/github/shivam25th/flyrank-1st/blob/main/work/notebooks/w04_signal_audit.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import os
import subprocess
from pathlib import Path

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

REPO_URL = "https://github.com/shivam25th/flyrank-1st.git"
REPO = Path("/content/flyrank-1st")

if not REPO.exists():
    subprocess.run(
        ["git", "clone", REPO_URL, str(REPO)],
        check=True
    )

os.chdir(REPO)

DATA_PATH = REPO / "data" / "raw" / "content_refresh_anonymized.csv"

assert DATA_PATH.exists(), f"CSV not found: {DATA_PATH}"

df = pd.read_csv(DATA_PATH)

df["is_declining_label"] = (
    df["trend_direction"].astype(str).str.lower() == "down"
).astype(int)

print("Repository:", REPO)
print("Dataset shape:", df.shape)

Repository: /content/flyrank-1st
Dataset shape: (30000, 45)


# ML-06 — Signal Audit: Do the Flags Hold?

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/shivam25th/flyrank-1st/blob/main/work/notebooks/w04_signal_audit.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Distributions

### Distribution review

The main traffic and search-performance variables are unevenly distributed. A small number of pages can have much larger search volume or impressions than the majority of pages.

Because of these heavy tails, raw averages and ordinary Pearson correlations can be misleading. I will use log-transformed traffic-like variables, grouped summaries, or rank-based comparisons where appropriate.

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
key_fields = [
    "search_volume",
    "impressions_90d",
    "clicks_90d",
    "sessions_90d",
    "word_count",
    "avg_position",
    "ctr"
]

summary = df[key_fields].describe().T[
    ["count", "mean", "50%", "std", "min", "max"]
]

print(summary.round(3))

print("\nTraffic-like variables: median vs mean")
for col in ["search_volume", "impressions_90d", "clicks_90d", "sessions_90d"]:
    print(
        f"{col:20s} "
        f"mean={df[col].mean():.2f} | "
        f"median={df[col].median():.2f}"
    )

                   count      mean      50%        std  min       max
search_volume    27532.0   158.882    10.00   1518.271  0.0   74000.0
impressions_90d  30000.0  5200.366   731.00  16838.020  1.0  517715.0
clicks_90d       30000.0    16.097     1.00     75.077  0.0    4178.0
sessions_90d     30000.0    37.067     7.00    107.069  1.0    4345.0
word_count       22301.0  3107.760  2877.00   1452.383  8.0    9546.0
avg_position     30000.0    16.342    10.80     15.217  0.0     245.0
ctr              30000.0     0.511     0.07      3.279  0.0     100.0

Traffic-like variables: median vs mean
search_volume        mean=158.88 | median=10.00
impressions_90d      mean=5200.37 | median=731.00
clicks_90d           mean=16.10 | median=1.00
sessions_90d         mean=37.07 | median=7.00


## 2. Signal test #1 / #2 / #3 (verdict each)

### Signal test #1 — Search volume and impressions

**Claim:** Pages associated with higher search volume should generally receive more impressions.

**Test:** Compare log-transformed search volume with log-transformed impressions using Spearman correlation and grouped medians.

**Verdict:** The result will be classified as CONFIRMED, OPPOSITE, MIXED, or FALSE based on the observed relationship.


### Verdict — Signal test #1

**Verdict: [CONFIRMED / OPPOSITE / MIXED / FALSE]**

The observed relationship between search volume and impressions is [describe what your output shows]. The grouped results contain sufficient observations in each bucket, so the result can be used as directional decision-support rather than causal evidence.


### Signal test #2 — Search position and CTR

**Claim:** Pages appearing in better search positions tend to have higher CTR.

**Test:** Compare median CTR and sample size across position tiers, restricting the analysis to pages with at least 100 impressions so that extremely low-exposure pages do not dominate the result.

**Verdict:** The result will be classified as CONFIRMED, OPPOSITE, MIXED, or FALSE.

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
signal1 = df[
    (df["search_volume"] >= 0) &
    (df["impressions_90d"] >= 0)
].copy()

signal1["log_search_volume"] = np.log1p(signal1["search_volume"])
signal1["log_impressions"] = np.log1p(signal1["impressions_90d"])

spearman_1 = signal1[
    ["search_volume", "impressions_90d"]
].corr(method="spearman").iloc[0, 1]

signal1["search_volume_bucket"] = pd.qcut(
    signal1["search_volume"].rank(method="first"),
    q=5,
    labels=["Q1", "Q2", "Q3", "Q4", "Q5"]
)

grouped_1 = (
    signal1
    .groupby("search_volume_bucket", observed=True)
    .agg(
        n=("impressions_90d", "size"),
        median_impressions=("impressions_90d", "median"),
        median_search_volume=("search_volume", "median")
    )
)

print("Spearman correlation:", round(spearman_1, 3))
print("\nGrouped results:")
print(grouped_1.round(2))

Spearman correlation: -0.029

Grouped results:
                         n  median_impressions  median_search_volume
search_volume_bucket                                                
Q1                    5507               989.0                   0.0
Q2                    5506              1006.5                   0.0
Q3                    5506               849.5                  10.0
Q4                    5506               904.5                  20.0
Q5                    5507               832.0                 110.0


In [ ]:
signal2 = df[
    (df["impressions_90d"] >= 100) &
    (df["avg_position"] > 0)
].copy()

position_groups = (
    signal2
    .groupby("position_tier", dropna=False)
    .agg(
        n=("ctr", "size"),
        median_ctr=("ctr", "median"),
        mean_ctr=("ctr", "mean"),
        median_position=("avg_position", "median")
    )
    .sort_values("median_position")
)

print(position_groups.round(4))

                  n  median_ctr  mean_ctr  median_position
position_tier                                             
top_3           533        0.19    0.3341              2.4
page_1         8633        0.23    0.3548              6.6
striking       5903        0.15    0.2558             14.0
page_3_5       6058        0.06    0.1424             28.8
deep            879        0.00    0.0554             59.5


In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
signal2 = df[
    (df["impressions_90d"] >= 100) &
    (df["avg_position"] > 0)
].copy()

position_groups = (
    signal2
    .groupby("position_tier", dropna=False)
    .agg(
        n=("ctr", "size"),
        median_ctr=("ctr", "median"),
        mean_ctr=("ctr", "mean"),
        median_position=("avg_position", "median")
    )
    .sort_values("median_position")
)

print(position_groups.round(4))

                  n  median_ctr  mean_ctr  median_position
position_tier                                             
top_3           533        0.19    0.3341              2.4
page_1         8633        0.23    0.3548              6.6
striking       5903        0.15    0.2558             14.0
page_3_5       6058        0.06    0.1424             28.8
deep            879        0.00    0.0554             59.5


In [ ]:
signal3 = df[
    df["content_age_days"].notna()
].copy()

signal3["age_bucket"] = pd.qcut(
    signal3["content_age_days"].rank(method="first"),
    q=5,
    labels=["Q1 youngest", "Q2", "Q3", "Q4", "Q5 oldest"]
)

age_results = (
    signal3
    .groupby("age_bucket", observed=True)
    .agg(
        n=("is_declining_label", "size"),
        median_age_days=("content_age_days", "median"),
        declining_rate=("is_declining_label", "mean")
    )
)

print(age_results.round(3))

                n  median_age_days  declining_rate
age_bucket                                        
Q1 youngest  6000            104.0           0.605
Q2           6000            144.0           0.647
Q3           6000            236.0           0.586
Q4           6000            321.0           0.449
Q5 oldest    6000            463.0           0.424


### Signal test #3 — Content age and decline

**Claim:** Older content may be more likely to belong to the defined declining group.

**Test:** Compare the declining rate across content-age buckets. Each bucket must have enough observations before interpreting the result.

**Verdict:** The result will be classified as CONFIRMED, OPPOSITE, MIXED, or FALSE.

### Verdict — Signal test #2

**Verdict: [CONFIRMED / OPPOSITE / MIXED / FALSE]**

The observed CTR pattern across position tiers is [your observation]. This is a directional association in the supplied data and does not establish that changing position alone causes the CTR change.

## 3. The flag-linked test

### Does the stale + visible flag identify more declining pages?

The baseline uses a `stale_visible_page` flag when a page has gone at least 180 days since its last update and has at least 500 impressions in the observed 90-day window.

I will compare the declining rate for flagged and unflagged pages. The comparison will include sample sizes so that a high rate from a very small group is not over-interpreted.

## 4. What this means in practice

*Two or three sentences: what a content team should take from this.*

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
df["stale_visible_flag"] = (
    (df["days_since_last_update"] >= 180) &
    (df["impressions_90d"] >= 500)
)

flag_test = (
    df
    .groupby("stale_visible_flag")
    .agg(
        n=("is_declining_label", "size"),
        declining_rate=("is_declining_label", "mean"),
        median_impressions=("impressions_90d", "median"),
        median_days_since_update=("days_since_last_update", "median")
    )
)

print(flag_test.round(3))

flagged_rate = df.loc[
    df["stale_visible_flag"],
    "is_declining_label"
].mean()

unflagged_rate = df.loc[
    ~df["stale_visible_flag"],
    "is_declining_label"
].mean()

print("\nFlagged declining rate:", round(flagged_rate, 3))
print("Unflagged declining rate:", round(unflagged_rate, 3))
print(
    "Difference:",
    round(flagged_rate - unflagged_rate, 3)
)

                        n  declining_rate  median_impressions  \
stale_visible_flag                                              
False               29983           0.542               731.0   
True                   17           0.941              4429.0   

                    median_days_since_update  
stale_visible_flag                            
False                                   20.0  
True                                   194.0  

Flagged declining rate: 0.941
Unflagged declining rate: 0.542
Difference: 0.399


In [ ]:
print("Signal audit completed.")
print("Rows analysed:", len(df))

print("\nKey fields checked:")
for col in [
    "search_volume",
    "impressions_90d",
    "avg_position",
    "ctr",
    "content_age_days",
    "days_since_last_update"
]:
    print("-", col)

print("\nFlagged pages:", int(df["stale_visible_flag"].sum()))
print("Unflagged pages:", int((~df["stale_visible_flag"]).sum()))

Signal audit completed.
Rows analysed: 30000

Key fields checked:
- search_volume
- impressions_90d
- avg_position
- ctr
- content_age_days
- days_since_last_update

Flagged pages: 17
Unflagged pages: 29983


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.